# Task 3: Correlation Analysis
This notebook analyzes the relationship between news sentiment and stock price returns.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Load data
news_df = pd.read_csv('../data/processed/news_sentiment.csv')
stock_df = pd.read_csv('../data/processed/stock_with_indicators.csv', header=[0, 1, 2], index_col=0)
stock_df.columns = stock_df.columns.get_level_values(0)
stock_df.index.name = 'Date'
stock_df = stock_df.reset_index()

# Normalize dates
news_df['date'] = pd.to_datetime(news_df['date']).dt.date
stock_df['Date'] = pd.to_datetime(stock_df['Date']).dt.date

# Aggregate sentiment
daily_sentiment = news_df.groupby(['date', 'stock'])['sentiment_avg'].mean().reset_index()

# Calculate returns
stock_df = stock_df.sort_values(['Date'])
price_col = 'Adj Close' if 'Adj Close' in stock_df.columns else 'Close'
stock_df['daily_return'] = stock_df[price_col].pct_change() * 100

# Merge
daily_sentiment_aapl = daily_sentiment[daily_sentiment['stock'] == 'AAPL']
merged_df = pd.merge(daily_sentiment_aapl, stock_df, left_on='date', right_on='Date')
merged_df.head()

## Correlation Analysis

In [ ]:
if len(merged_df) > 1:
    corr, p_value = pearsonr(merged_df['sentiment_avg'], merged_df['daily_return'])
    print(f"Pearson Correlation: {corr:.4f}")
    print(f"P-value: {p_value:.4f}")

plt.figure(figsize=(10, 6))
sns.regplot(x='sentiment_avg', y='daily_return', data=merged_df)
plt.title('Sentiment vs Daily Return')
plt.show()